# Stellar Neural Network Interpolation - ShinkaEvolve Launcher

**Task**: Optimize a FLAX neural network for stellar evolution track interpolation
**Objective**: Max fractional residual for Delta Nu < 0.00079 (Kepler-LEGACY)

## Prerequisites
Before running this notebook, import the following packages in the virtual environment. See the README of the repo to see how to add packages to the virtual environment.
- jax
- flax
- tables

# Part 1. Pre-flight Check

Before we begin, let's verify that our programming environment is set up correctly. This notebook should be running via a JupyterLab server executed in a virtual environment. The following block of code will do two things.

1.  Check that your virtual environment has the Python ShinkaEvolve package `shinka` installed.

2.  Load the **OpenRouter API key** into the Jupyter notebook.

**Important (!)** - Make sure that your OpenRouter API key is contained a `.env` file located at the root of this project, i.e. immediately inside the `tutorial_shinka/` directory.

In [ ]:
import warnings
import dotenv
import os

# Suppress third-party warnings before importing shinka
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

# Check if ShinkaEvolve is importable
try:
    import shinka
except ImportError:
    print("shinka not found, make sure to open this notebook with the ShinkaEvolve environment active")

# Find the .env file
env_path = dotenv.find_dotenv()

# Make sure that there's a .env file
assert env_path, ".env not found, please add it to the root of this project."

# Find the repository root assuming that's where the .env file is
repo_path = os.path.dirname(env_path)
activate_path = os.path.join(repo_path, '.venv/bin/activate')

# Print out where the .env file and repo root are
print("> loaded .env from {}".format(env_path))
print("> repo root located at {}".format(repo_path))

# Load the environment variables
dotenv.load_dotenv()

# Check that OPENROUTER_API_KEY is set in the .env file
if os.environ.get("OPENROUTER_API_KEY"):
    print("> OPENROUTER_API_KEY found")
else:
    print("> WARNING: OPENROUTER_API_KEY not set — add it to .env file")

We can verify that `evaluate.py` runs correctly on `initial_program.py` before launching evolution.

In [ ]:
import numpy as np
import evaluate
import initial_program

# Test the initial program for circle packing
data_bundles = evaluate.load_and_prep_data()
output = initial_program.run_training(**data_bundles)

# Check if the the program outputs a valid circle packing
valid, msg = evaluate.validate_stellar(output)

# Assert check
assert valid, f"> Initial test FAILED: {msg}"

val_loss = output["val_loss"]
max_res = np.max(output["test_residuals"][:,2]) # I think this is the right formula, please double check

print(f"> Initial test: PASSED  (loss={val_loss:.6f}, max_residual={max_res:.6f})")

# Part 2. Configuring the ShinkaEvolve Experiment

There are a number of hyperparameters that one can use to configure a ShinkaEvolve experiment. There are three Python objects which collect all possible parameters that can be used to configure a ShinkaEvolve experiment.

-   `EvolutionConfig`

-   `DatabaseConfig`

-   `LocalJobConfig`

The **bare minimum** collection of parameters that the user needs to define in order to run an experiment are the following.

-   `init_program_path` in `EvolutionConfig` - This parameter tells ShinkaEvolve where your **initial program** is. The initial program is the first program to be added to the population, and will be used to evolve subsequent programs.

-   `eval_program_path` in `LocalJobConfig` - This parameter tells ShinkaEvolve where your **evaluation code**. The evaluation code contains all code **outside the context of an LLM** that you write for (1) validating that generated programs are correct, and (2) evaluating the score of the program.

Let's first define these parameters as global variables.

In [ ]:
import datetime as dt

# A description of the task ShinkaEvolve is solving to be sent as an LLM prompt.
TASK_SYS_MSG = """
Your task is to perform interpolation using a jax-based neural network (FLAX) over a grid of stellar evolution tracks generated by ASTEC. 
The goal is to achieve absolute-valued fractional residuals for the test set below 0.00079 for 'delta_nu', while keeping validation loss small.

Key directions to explore:
1. Various hyperparameter values (learning rate, weight decay, dropout, learning rate scheduler, etc.)
2. Various architectures, incuding changing the batch size, hidden dimensions, neural network layer types, etc.
3. You can try k-fold validation to improve

Constraints:
1. Make sure that you use a jax-based neural network (FLAX) and not another type of machine-learning interpolator.
2. The program will be evaluated on a CPU with only a few cores available. Please be aware of this as to avoid proposing architectures 
   that are too complex and require a long time to train.
3. The name of the Python class representing the neural network must be StellarNet, even if its content changes

Be creative!
"""

# Number of generations to run in this experiment
NUM_GENERATIONS = 30

# Results will be stored in a directory "circpack_<CURRENT DATE-TIME>"
experiment_name = "stellar_" + dt.datetime.now().strftime("%Y%m%d_%H%M%S")

# Set RESULTS_DIR
RESULTS_DIR = "results/{}".format(experiment_name)

# Print out the path
print(f"> Results dir: {RESULTS_DIR}")

In [ ]:
# Set cost if you want to try this out!
cost = 5

# Define the MAX_API_COST variable
MAX_API_COST = cost or None

# Has my cost been set?
print('> Cost limit? {}'.format(MAX_API_COST))

# Define all LLM related hyperparameters.
LLM_MODELS = [
    "openrouter/anthropic/claude-haiku-4-5",
    "openrouter/openai/gpt-5.2-codex",
    "openrouter/google/gemini-3-flash-preview",
    "openrouter/openai/gpt-5-nano"
]

META_LLM_MODELS = ["openrouter/openai/o4-mini"]

NOVELTY_LLM_MODELS = ["openrouter/openai/o4-mini"]

EMBEDDING_MODEL = "openrouter/openai/text-embedding-3-small"

In [ ]:
# Import from config objects from the shinka library
from shinka.core import EvolutionConfig
from shinka.database import DatabaseConfig
from shinka.launch import LocalJobConfig


# Set the evolution config object
evo_config = EvolutionConfig(init_program_path="initial_program.py",
                             task_sys_msg=TASK_SYS_MSG,
                             num_generations=NUM_GENERATIONS,
                             results_dir=RESULTS_DIR,
                             max_api_costs=MAX_API_COST,
                             llm_models=LLM_MODELS,
                             meta_llm_models=META_LLM_MODELS,
                             novelty_llm_models=NOVELTY_LLM_MODELS,
                             embedding_model=EMBEDDING_MODEL)

# Number of islands is a hyperparameter which affects the evolution algorithm
# run by ShinkaEvolve. This also affects the visualization. See the guide
# at `recipes/hyperparameters.md` for more information.
db_config = DatabaseConfig(num_islands=3)

The following cell is the only part of the notebook that is different if you are working with Conda or Python virtual environments.
- **Conda**: uncomment the `job_config` setup that sets `conda_env = "shinka_ai4sd"` (the name of the environment will work on Bouchet for this tutorial, otherwise use the name of your Conda environment)
- **Python**: uncomment the `job_config` setup that sets `activate_script = activate_path`. This was defined in Part 1, and it assumes that `.venv` for the desired environment lives in the `tutorial_shinka` folder, i.e. the Python executable is at `tutorial_shinka/.venv/bin/python`.

In [ ]:
# Set the job config. ACTIVATE_SCRIPT is a parameter which tells what virtual
# environment ShinkaEvolve will run evolved programs in.

# job_config = LocalJobConfig(eval_program_path=EVAL_PROGRAM_PATH,
#                             activate_script=activate_path)

# If you're using Conda to manage your virtual environment, 
# you will need to set CONDA_ENV instead.

job_config = LocalJobConfig(eval_program_path=EVAL_PROGRAM_PATH,
                            conda_env="shinka_ai4sd26")

# Part 3. Launch ShinkaEvolve

Now we're ready to launch ShinkaEvolve. Pass all config parameters (the `EvolutionConfig, DatabaseConfig, LocalJobConfig` objects) to a `ShinkaEvolveRunner` object. Then call `run_async` to start the evolving.

**IMPORTANT (!)** - Running the next block will output a lot of text! You can continue on to **Part 4** as this block runs.

In [ ]:
from shinka.core import ShinkaEvolveRunner

from time import perf_counter

MAX_PROPOSAL_JOBS = 4
MAX_EVALUATION_JOBS = 3

# Create the ShinkaEvolveRunner object.
runner = ShinkaEvolveRunner(
    evo_config=evo_config,
    job_config=job_config,
    db_config=db_config,
    max_proposal_jobs=MAX_PROPOSAL_JOBS,
    max_evaluation_jobs=MAX_EVALUATION_JOBS,
    verbose=True,
)

# Store the starting wallclock time
tic = perf_counter()

# Run ShinkaEvolve by calling `run_async`
await runner.run_async()

# Store the stop wallclock time
toc = perf_counter()

# Print out information
print(f"> Evolution completed in {toc - tic:.1f} s")
print(f"> Results saved to: {runner.results_dir}")

# Part 4. Visualizing ShinkaEvolve using the WebUI

By default, logging information when running ShinkaEvolve will be sent to the environment you're running your code in (Jupyterlab or the terminal). This text can be hard to parse, and so the ShinkaEvolve package also contains a **visualization tool** that has a **web user interface**. 

You can launch this tool through the command line by following these steps.

1.  Open a separate terminal, navigate to the directory containing this repository `tutorial_shinka`.

2.  Activate the virtual environment.

3.  Run the following command inside the repository

    ```bash
    shinka_visualize --db results --port 8000 --open
    ```

This will open the WebUI with a lot of information about the evolution.

**IMPORTANT (!)** - There may be a bug when launching the Web UI, where the browser tab opens but the database is not loaded. To resolve it, click on `Dashboard` on the top left corner and the list of available databases should appear.